# Using LLM prompt templates and artifacts

This tutorial illustrates how easy it is to use LLMs and prompt templates, inside a complete workflow using the llm-prompt artifact.

Whenever an LLM-Prompt artifact is used, there MUST be a definition of:
- What is the prompt template
- Which LLM is used
- What the model’s generation configuration is (if not using the default)

The model we are using is `gpt-4o-mini` from OpenAI with the default configuration (see section 3 for available model params), this case covers using a remote model directly from the configured datasource without having to download it first.
We use streamlit to create a chat front-end and deploy it as [application runtime](https://docs.mlrun.org/en/latest/runtimes/application.html).

**In this tutorial**
* [Set up the environment](#Set-up-the-environment)
* [Import mlrun library and initialize the project](#Import-mlrun-library-and-initialize-the-project)
* [Configure OpenAI profile](#Configure-OpenAI-profile)
* [Define the LLM and the prompt templates](#Define-the-LLM-and-the-prompt-templates)
* [Define the function graph, adding ModelRunnerStep with proxy models for the shared model](#Define-the-function-graph,-adding-ModelRunnerStep-with-proxy-models-for-the-shared-model)
* [Enable tracking, visualize the graph, and deploy the function](#Enable-tracking,-visualize-the-graph,-and-deploy-the-function)
* [Deploy the model monitoring application](#Deploy-the-model-monitoring-application)
* [Configure the Streamlit chatbot application](#Configure-the-Streamlit-chatbot-application)
* [Launch the Streamlit Chatbot to Interact with the LLM Model](#Launch-the-Streamlit-Chatbot-to-Interact-with-the-LLM-Model)

In [ ]:
!pip install streamlit

## Set up the environment
This section sets up the environment variables required for OpenAI API access, including the base URL and API key.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("ai_gateway.env")

assert (os.environ.get("OPENAI_API_KEY", None) is not None) and (
    os.environ.get("OPENAI_BASE_URL", None) is not None
), "\
    Missing OpenAI credentials, make sure they are set as environment variables."
os.environ["OPENAI_MAX_RETRIES"] = "100"

## Import mlrun library and initialize the project
This initializes the MLRun project

In [ ]:
%config Completer.use_jedi = False

import mlrun
from mlrun import get_or_create_project

image = "mlrun/mlrun"
project_name = "llm-openai-bot"
project = get_or_create_project(project_name, context="./")

This section sets up the necessary datastore profiles for time-series database (TSDB) and stream data.
which are essential for monitoring model performance and detecting drift.
You can use a data store profile to manage datastore credentials.
A data store profile holds all the information required to address an external data source, including credentials.
The `DatastoreProfileV3io` is used for V3IO storages while `DatastoreProfileTDEngine`, `DatastoreProfileKafkaSource` are used in community edition.
Notice that recommended base period is 10 minutes, for demo purposes we set base period to 1 minute.

In [ ]:
from src.model_monitoring_utils import enable_model_monitoring

enable_model_monitoring(
    project=project, deploy_histogram_data_drift_app=False, base_period=1
)

## Configure OpenAI profile
This section sets up an openAI profile (credentials and environment variables), and specifies the model. This tutorial uses the model `gpt-4o-mini`. You can change it to any model you want to use.

In [ ]:
from mlrun.datastore.datastore_profile import OpenAIProfile

open_ai_profile = OpenAIProfile(
    name="openai_profile",
    api_key=os.environ.get("OPENAI_API_KEY"),
    organization=os.environ.get("OPENAI_ORG_ID"),
    project=os.environ.get("OPENAI_PROJECT_ID"),
    base_url=os.environ.get("OPENAI_BASE_URL"),
    timeout=os.environ.get("OPENAI_TIMEOUT"),
    max_retries=os.environ.get("OPENAI_MAX_RETRIES"),
)
project.register_datastore_profile(open_ai_profile)
model_url = f"ds://openai_profile/gpt-4o-mini"

## Define the LLM and the prompt templates
This section defines the LLM and the prompt templates for finance and sport domains. The `finance_prompt_template` and `sport_prompt_template` ([`src/llm_prompts.py`](./src/llm_prompts.py)) are structured to guide the LLM in generating responses based on user queries. The templates include a system message that sets the context for the LLM, and a user message that includes the user's ID, tone, depth level, and question.

In [ ]:
from src.llm_prompts import finance_prompt_template, sport_prompt_template

model_artifact = project.log_model(
    "open-ai",
    model_url=model_url,
)
finance_llm_prompt_artifact = project.log_llm_prompt(
    "finance_llm_prompt",
    prompt_template=finance_prompt_template,
    model_artifact=model_artifact,
    prompt_legend={
        "question": {
            "field": "question",
            "description": "The main financial question or request the user is asking.",
        },
        "depth_level": {
            "field": "depth_level",
            "description": "Indicates the level of detail in the answer (e.g., basic, intermediate, advanced).",
        },
        "user_id": {
            "field": "user_id",
            "description": "Unique identifier of the user, useful for personalization and tracking.",
        },
        "tone": {
            "field": "tone",
            "description": "The desired style of the response (e.g., formal, friendly, concise, detailed).",
        },
    },
)
sport_llm_prompt_artifact = project.log_llm_prompt(
    "sport_llm_prompt",
    prompt_template=sport_prompt_template,
    model_artifact=model_artifact,
    prompt_legend={
        "question": {
            "field": "question",
            "description": "The main sports or fitness-related question from the user.",
        },
        "depth_level": {
            "field": "depth_level",
            "description": "Indicates how in-depth the explanation should be (e.g., beginner, intermediate, expert).",
        },
        "user_id": {
            "field": "user_id",
            "description": "Unique identifier of the user, used for personalization or tracking.",
        },
        "tone": {
            "field": "tone",
            "description": "The preferred style or tone of the response (e.g., motivational, professional, casual).",
        },
    },
)

## Define the function graph, adding ModelRunnerStep with proxy models for the shared model
Model runner step is used to run multiple models on each event.
When `ModelRunnerStep` is used in a graph, MLRun automatically imports the default language model class (LLModel) during function deployment.
Use the `add_shared_model` to add a shared model to the graph, this model will be available to all the `ModelRunners` in the graph, and `add_shared_model_proxy` to add a proxy model to the ModelRunnerStep, which is a proxy for a model that is already defined as shared model within the graph.

In [ ]:
from mlrun.serving import ModelRunnerStep
from mlrun.common.schemas.model_monitoring.constants import (
    ModelEndpointCreationStrategy,
)

function = mlrun.code_to_function(
    name="open-ai-tut",
    kind="serving",
    tag="latest",
    project=project.name,
    filename="./src/LLM_file.py",
    image=image,
    requirements=["openai==1.77.0"],
)
graph = function.set_topology("flow", engine="async")

model_runner_step = ModelRunnerStep(
    name="model_runner_step", model_selector="MyModelSelector"
)

graph.add_shared_model(
    name="shared_llm",
    execution_mechanism="dedicated_process",
    model_class="LLModel",
    model_artifact=model_artifact,
    result_path="outputs",
)

model_runner_step.add_shared_model_proxy(
    endpoint_name="finance_endpoint",
    model_artifact=finance_llm_prompt_artifact,
    shared_model_name="shared_llm",
    model_endpoint_creation_strategy=ModelEndpointCreationStrategy.OVERWRITE,
)
model_runner_step.add_shared_model_proxy(
    endpoint_name="sport_endpoint",
    model_artifact=sport_llm_prompt_artifact,
    shared_model_name="shared_llm",
    model_endpoint_creation_strategy=ModelEndpointCreationStrategy.OVERWRITE,
)

graph.to(model_runner_step).respond()

## Enable tracking, visualize the graph, and deploy the function
This section enables experiment tracking, deploys the function, and visualizes the workflow of the LLM model using a graph within the Streamlit app.
**Note:** The `deploy_endpoint` provides the URL to interact with the Streamlit interface.

In [ ]:
function.set_tracking(enable_tracking=True)
graph.plot()

In [ ]:
deploy_endpoint = function.deploy()

## Deploy the model monitoring application
This section deploys the model monitoring application, which is responsible for monitoring the performance of the LLMs that were deployed in the previous step. It uses the monitoring_application.py script to define the monitoring logic. The application is deployed using the `deploy_function` method, which makes it available for monitoring the LLMs in real time.

In [ ]:
llm_monitoring_app = project.set_model_monitoring_function(
    func="./src/monitoring_application.py",
    application_class="ModelMonitoringApplication",
    name="llm-monitoring",
    image=image,
)

project.deploy_function(llm_monitoring_app)

## Configure the Streamlit chatbot application
This section sets up a Streamlit app that enables you to interact with the LLMs deployed in the previous steps. The app provides a user interface for selecting different models, tones, and depth levels, and allows users to submit questions to the LLMs.


In [ ]:
%%writefile streamlit_ui.py

import os
import json
from typing import Any, Dict, List

import requests
import streamlit as st

st.set_page_config(page_title="LLM Playground", layout="wide")

API_URL = os.getenv("API_URL")  # e.g. http://localhost:8000/v1/generate
assert API_URL, "API_URL not set"

# Options (from your Gradio code)
PROMPT_OPTIONS = ["finance_endpoint", "sport_endpoint"]
TONE_OPTIONS = ["formal", "casual", "optimistic", "neutral"]
DEPTH_LEVELS = ["basic overview", "detailed explanation", "expert-level analysis"]
USER_ID = 12345


def generate(
    model_name: str, tone: str, depth: str, prompt: str
) -> str:
    """API call for model generation."""
    payload = {
        "model_name": model_name,
        "question": prompt,
        "depth_level": depth,
        "user_id": USER_ID,
        "tone": tone,
    }
    resp = requests.post(API_URL, data=json.dumps(payload).encode("utf-8"))
    resp.raise_for_status()
    resp_json = resp.json()
    return (
        resp_json.get(model_name, {})
        .get("outputs", {})
        .get("answer", "No response available.")
    )


def ensure_state() -> None:
    """Initialize session state variables if not already set."""
    defaults = {
        "messages": [],
        "model_name": PROMPT_OPTIONS[0],
        "tone": TONE_OPTIONS[0],
        "depth": DEPTH_LEVELS[0],
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v


def render_sidebar() -> None:
    """Sidebar controls for parameters and clearing chat."""
    with st.container(height=600):
        st.write("#### Parameters")
        st.session_state.model_name = st.selectbox(
            "Select Model",
            options=PROMPT_OPTIONS,
            index=PROMPT_OPTIONS.index(st.session_state.model_name),
        )
        st.session_state.tone = st.selectbox(
            "Select Tone",
            options=TONE_OPTIONS,
            index=TONE_OPTIONS.index(st.session_state.tone),
        )
        st.session_state.depth = st.selectbox(
            "Select Depth",
            options=DEPTH_LEVELS,
            index=DEPTH_LEVELS.index(st.session_state.depth),
        )

        if st.button("Clear Chat", use_container_width=True):
            st.session_state.messages = []
            st.rerun()


def render_chat():
    """Render the chat interface and handle user input."""
    with st.container(height=600):
        messages = st.container(height=500)

        # Render prior chat
        for m in st.session_state.messages:
            role = m.get("role", "assistant")
            content = m.get("content", "")
            messages.chat_message(role).write(content)

        # Chat input
        user_prompt = st.chat_input("Type a question:")
        if user_prompt:
            # Show user message
            st.session_state.messages.append({"role": "user", "content": user_prompt})
            messages.chat_message("user").write(user_prompt)

            # Call backend
            with messages:
                with st.spinner("Generating response..."):
                    bot_message = generate(
                        st.session_state.model_name,
                        st.session_state.tone,
                        st.session_state.depth,
                        user_prompt,
                    )

            # Show assistant message
            st.session_state.messages.append(
                {"role": "assistant", "content": bot_message}
            )
            with messages.chat_message("assistant"):
                st.write(bot_message)

            st.rerun()


def main():
    ensure_state()
    st.write("# 🤖 LLM Playground with Model Selector")
    left, right = st.columns([3, 1], gap="small")
    with right:
        render_sidebar()
    with left:
        render_chat()


if __name__ == "__main__":
    main()

In [ ]:
!tar -czvf frontend_ui.tar.gz ./streamlit_ui.py

In [ ]:
# Log the streamlit tar file as project artifact and use it as source archive
frontend_source = project.log_artifact(
    "frontend_source", local_path="./frontend_ui.tar.gz", upload=True
)

ui_fn = project.set_function(
    name="frontend",
    kind="application",
    image="mlrun/mlrun",
    requirements=["streamlit==1.49.1"],
)


API_URL = function.get_url()

# Set application spec and envs
ui_fn.set_env("API_URL", API_URL)
ui_fn.with_source_archive(frontend_source.target_path, pull_at_runtime=False)
ui_fn.set_internal_application_port(8000)
ui_fn.spec.command = "streamlit"
ui_fn.spec.args = ["run", "--server.port", "8000", "/home/mlrun_code/streamlit_ui.py"]

## Launch the Streamlit Chatbot to Interact with the LLM Model
This section launches the Streamlit chatbot, providing a user-friendly interface for interacting with the deployed LLM models. Users can select the model, tone, and depth level, submit questions, and view responses in a chat-style format.


In [ ]:
ui_fn.deploy(with_mlrun=False, create_default_api_gateway=False)
ui_fn.create_api_gateway(
    name="llm-prompt-artifact-ui",
    path="/",
    direct_port_access=True,
    ssl_redirect=True,
    set_as_default=False,
    authentication_mode="none",
)

In [ ]:
print(
    f"Use this address to interact with your new chatbot ! https://{ui_fn.status.address}"
)

![Model Architecture](./_static/images/llm-prompt-streamlit-ui.png)